# Silicon Sampling Phase 4: Track B Fine-Tuning

**Objective:** Fine-tune Llama-3.1-8B-Instruct on real Indian WVS responses using QLoRA 4-bit quantization.

**Compute:** Kaggle T4 GPU, ~3-4 GPU-hours per fold, 5 folds ≈ 20 GPU-hours

**Output:** Fine-tuned models per fold, with out-of-fold predictions for all N respondents.

**Key features:**
- QLoRA 4-bit to fit 8B model on 16GB T4
- Training uses exact P2 prompt format + real answer
- Respondent-level stratified split (no test leakage)
- Checkpointing to Kaggle datasets for crash recovery
- Cross-validation: 5 folds, each fold gets tuned model and OOF predictions


## Setup & Dependencies

In [ ]:
import json
import logging
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(message)s')
logger = logging.getLogger(__name__)

# Add repo to path
import sys
sys.path.insert(0, '/kaggle/input/silicon-sampling-code')  # Adjust path

from src.config import DATA_PROCESSED, CACHE_DIR, COUNTRY_CODE
from src.prompts import build_prompt

logger.info("✓ Imports successful")
logger.info(f"PyTorch version: {torch.__version__}")
logger.info(f"CUDA available: {torch.cuda.is_available()}")

## Load Data & Configuration

In [ ]:
# Load processed data
data_path = DATA_PROCESSED / "ind_wvs7.parquet"
df = pd.read_parquet(data_path)
logger.info(f"Loaded {len(df)} respondents")

# Load selected items
items_path = DATA_PROCESSED / "selected_items.json"
with open(items_path) as f:
    selected_items = json.load(f)
logger.info(f"Loaded {len(selected_items)} selected items")

# Load fold configuration
folds_path = DATA_PROCESSED / "folds.json"
with open(folds_path) as f:
    fold_config = json.load(f)
logger.info(f"Loaded {fold_config['n_folds']}-fold CV configuration")

print(f"\nData shape: {df.shape}")
print(f"Items: {len(selected_items)}")
print(f"Training examples per fold: ~{len(df) * len(selected_items)}")

## Build Training Dataset (P2 Prompt Format)

In [ ]:
# Demographics columns
demo_cols = [
    "Q260", "Q262", "Q273", "Q274", "Q275", "Q279",
    "Q281", "Q287", "Q288", "Q289", "H_URBRURAL",
    "N_REGION_ISO", "G_TOWNSIZE", "LNGE_ISO",
]

def build_training_dataset(df_train: pd.DataFrame, df_test_ids: List[int]) -> List[Dict]:
    """
    Build training examples: (P2 prompt + true answer) for all train respondents × items.
    
    Returns list of {"prompt": ..., "response": ...} dicts for SFTTrainer.
    """
    examples = []
    
    for _, row in df_train.iterrows():
        respondent_id = int(row["respondent_id"])
        if respondent_id in df_test_ids:
            continue  # Skip test respondents
        
        # Extract demographics
        demographics = {col: row[col] for col in demo_cols if col in row}
        
        # For each item, create (prompt, answer) pair
        for item in selected_items:
            if item not in df_train.columns:
                continue
            
            true_answer = int(row[item])
            if pd.isna(true_answer):
                continue
            
            # Build P2 prompt (full structured demographics)
            question = f"Question {item}"  # Use real question text if available
            answer_options = ["1", "2", "3", "4"]  # Adjust based on item scale
            
            prompt = build_prompt(
                "P2",  # Full structured condition
                question,
                "\n".join(f"{i+1}. {opt}" for i, opt in enumerate(answer_options)),
                **demographics
            )
            
            # Format for SFTTrainer
            response = str(true_answer)
            full_text = prompt + "\n\nAnswer: " + response
            
            examples.append({
                "text": full_text,
                "respondent_id": respondent_id,
                "item_id": item,
            })
    
    return examples

logger.info("Training dataset builder ready")

## Initialize Model with 4-Bit Quantization

In [ ]:
model_name = "meta-llama/Llama-3.1-8B-Instruct"  # Same as Track A base

logger.info(f"Loading {model_name} with 4-bit quantization...")

# 4-bit quantization config for efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False  # Disable cache for training

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    padding_side="right",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

logger.info("✓ Model and tokenizer loaded")
logger.info(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Configure QLoRA

In [ ]:
# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=16,  # LoRA rank
    lora_alpha=32,  # LoRA scaling
    target_modules=["q_proj", "v_proj"],  # Target attention modules
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA
model = get_peft_model(model, lora_config)
logger.info("✓ QLoRA configured")

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
logger.info(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")

## Phase 4: Fine-Tuning Loop (5 Folds)

In [ ]:
logger.info(f"\n" + "="*60)
logger.info("PHASE 4: TRACK B FINE-TUNING")
logger.info("="*60)

finetune_results = []

# Get fold configuration
n_folds = fold_config["n_folds"]
folds = fold_config["folds"]

# Demo: run first fold only (set to n_folds for full run)
demo_folds = 1

for fold_idx in range(demo_folds):
    fold = folds[fold_idx]
    
    logger.info(f"\n--- FOLD {fold_idx + 1}/{n_folds} ---")
    
    train_ids = fold["train"]
    test_ids = fold["test"]
    
    logger.info(f"Train: {len(train_ids)} respondents")
    logger.info(f"Test: {len(test_ids)} respondents")
    
    # Split data
    df_train = df[df["respondent_id"].isin(train_ids)].copy()
    df_test = df[df["respondent_id"].isin(test_ids)].copy()
    
    # Build training dataset
    logger.info("Building training dataset...")
    train_examples = build_training_dataset(df_train, test_ids)
    logger.info(f"Created {len(train_examples):,} training examples")
    
    if len(train_examples) == 0:
        logger.error("No training examples created. Check data and prompts.")
        continue
    
    # Training arguments
    output_dir = f"/kaggle/working/checkpoint_fold_{fold_idx}"
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=2,
        per_device_train_batch_size=4,  # Adjust based on GPU memory
        gradient_accumulation_steps=4,
        warmup_steps=100,
        learning_rate=2e-4,
        weight_decay=0.01,
        fp16=True,
        logging_steps=50,
        save_steps=500,
        save_total_limit=2,  # Keep only 2 checkpoints
        optim="paged_adamw_32bit",
        seed=42,
        max_grad_norm=1.0,
        remove_unused_columns=False,
        report_to="none",  # No WandB logging on Kaggle
    )
    
    logger.info("Initializing SFTTrainer...")
    
    # SFT Trainer
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_examples,  # List of dicts with "text" key
        args=training_args,
        packing=False,  # Disable packing for clarity
        max_seq_length=512,
        tokenizer=tokenizer,
        formatting_func=lambda x: {"text": x["text"]},
    )
    
    # Train
    logger.info(f"Training fold {fold_idx + 1}/{n_folds}...")
    trainer.train()
    
    # Save model
    model_path = f"/kaggle/working/model_fold_{fold_idx}"
    trainer.model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    logger.info(f"✓ Model saved to {model_path}")
    
    # Log results
    fold_result = {
        "fold": fold_idx,
        "n_train": len(train_ids),
        "n_test": len(test_ids),
        "n_train_examples": len(train_examples),
        "model_path": model_path,
    }
    finetune_results.append(fold_result)
    
    logger.info(f"Fold {fold_idx + 1} complete")

logger.info(f"\n" + "="*60)
logger.info(f"Fine-tuning of {demo_folds} fold(s) complete")
logger.info("="*60)

## Evaluate Fine-Tuned Model on Test Set

In [ ]:
# Load fine-tuned model and run inference on test set
# This verifies that fine-tuning improved the model

if finetune_results:
    fold_result = finetune_results[0]  # First fold
    model_path = fold_result["model_path"]
    
    logger.info(f"Loading fine-tuned model from {model_path}")
    
    # Load fine-tuned model
    from transformers import AutoModelForCausalLM
    ft_model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        trust_remote_code=True,
    )
    ft_tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    logger.info("✓ Fine-tuned model loaded")
    
    # Sample test prediction
    # In production, run full evaluation on test set

## Verify No Leakage

In [ ]:
# Verify that no test respondent appears in training
logger.info("\nVerifying no train/test leakage...")

for fold_idx, fold in enumerate(folds[:demo_folds]):
    train_ids = set(fold["train"])
    test_ids = set(fold["test"])
    
    overlap = train_ids & test_ids
    assert len(overlap) == 0, f"Fold {fold_idx}: {len(overlap)} respondents in both train and test!"
    
    logger.info(f"Fold {fold_idx}: ✓ No overlap")

logger.info("✓ All folds verified")

## Next Steps

1. **Run full fine-tuning** (set `demo_folds = n_folds` and run)
   - ~3-4 GPU-hours per fold on T4
   - All 5 folds ≈ 20 GPU-hours
   
2. **Generate out-of-fold predictions**
   - For each fold: load fine-tuned model and predict on test respondents
   - Save predictions (identical format to Track A)
   
3. **Checkpoint to Kaggle Datasets**
   - Save fine-tuned model weights after each fold
   - Ensures no work lost if session crashes
   
4. **Phase 5: Subgroup Analysis**
   - Compare zero-shot (Track A) vs fine-tuned (Track B) on each subgroup
   - Compute fidelity gap and Δ_gap
   - Bootstrap CIs on all numbers

In [ ]:
logger.info(f"\n" + "="*60)
logger.info("Phase 4 template complete!")
logger.info("="*60)
logger.info(f"Fine-tuned {len(finetune_results)} fold(s)")
logger.info(f"Ready for Phase 5 analysis")